# Capstone Project · Automated Gene Report & Structural Analysis Pipeline
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/Capstone/Capstone_Gene_Report_Pipeline.ipynb)

⏱ ~8 hours · Difficulty: ★★★ · Prerequisites: Modules M01 through M06

---

## 1. Learning Outcomes

By completing this Capstone project, you will demonstrate mastery across all six core curriculum modules:
1. **Architect an end-to-end bioinformatics analysis pipeline** that ingests a gene symbol (e.g. `TP53`, `BRCA1`), coordinates multi-database queries, performs sequence and biophysical computations, generates publication figures, and compiles automated markdown reports.
2. **Implement professional Object-Oriented Programming (OOP) principles** (encapsulation, private/protected state, read-only properties, method delegation) to model complex genomic entities cleanly and defensively.
3. **Interface robustly with external bioinformatics APIs** (NCBI Entrez, UniProt REST) incorporating API authentication from `.env`, strict rate-limiting, error handling, and offline caching.
4. **Extract genomic features and annotations** from GenBank records using Biopython `SeqIO` and `SeqFeature` (CDS coordinates, exon spans, reading frames, and translations).
5. **Compute proteomic and biophysical profiles** including molecular mass, isoelectric point ($pI$), in vitro instability index, hydropathy (GRAVY), and secondary structure fractions using `Bio.SeqUtils.ProtParam`.
6. **Perform 3D macromolecular structural analysis** by parsing PDB coordinate files (`Bio.PDB`), computing vectorized Euclidean residue distance matrices, and plotting tertiary contact maps.
7. **Generate publication-ready visualizations** (exon models, rolling GC profiles, amino acid distribution, and contact maps) and export an automated Markdown report alongside an interactive HTML Gene Card.

---

## 2. Prerequisites

This Capstone synthesizes concepts from all course modules:
- **Module 1 (Python Basics):** Strings, collections, functions, and sequence calculations.
- **Module 2 (File Handling & Biopython):** `SeqIO`, GenBank parsing, `SeqFeature` locations.
- **Module 3 (OOP):** Classes, `self`, protected `_attr` vs private `__attr`, `@property` decorators.
- **Module 4 (Data Visualisation):** Matplotlib patches, multi-panel axes, and Seaborn themes.
- **Module 5 (Databases & Web APIs):** NCBI Entrez E-utilities, UniProt REST API, and `.env` secret management.
- **Module 6 (os, NumPy & pandas):** `Path.mkdir`, vectorization, tabular wrangling, and tidy reshaping.


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os, time, json, re, base64
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import Bio
from Bio import SeqIO, Entrez
from Bio.SeqUtils import gc_fraction, molecular_weight
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.PDB import PDBParser
import requests
from dotenv import load_dotenv

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn python-dotenv requests
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    current = Path.cwd().resolve()
    ROOT = None
    for p in [current, *current.parents]:
        if (p / "COURSE_PLAN.md").exists() or (p / ".git").exists():
            ROOT = p
            break
    if ROOT is None:
        ROOT = current

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "Capstone"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "cache").mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 150

print(f"Python:    {sys.version.split()[0]}")
print(f"Biopython: {Bio.__version__}")
print(f"NumPy:     {np.__version__}")
print(f"pandas:    {pd.__version__}")
print(f"ROOT:      {ROOT.resolve()}")
print(f"DATA_DIR:  {DATA_DIR.resolve()} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:   {OUT_DIR.resolve()}")


---

## 3. Pipeline Architecture & Design Overview

```
┌────────────────────────────────────────────────────────────────────────┐
│                   Enriched GeneReport Pipeline Flow                    │
│                                                                        │
│   [ Input: Gene Symbol (e.g. 'TP53') ]                                 │
│                   │                                                    │
│                   ▼                                                    │
│   Step 1: Ingestion & Caching (Entrez API / Local GenBank Cache)       │
│                   │                                                    │
│                   ▼                                                    │
│   Step 2: Genomic Feature Extraction (Biopython SeqIO -> Exons & CDS)  │
│                   │                                                    │
│                   ▼                                                    │
│   Step 3: Biophysical Protein Profiling (ProtParam -> MW, pI, Stability)│
│                   │                                                    │
│                   ▼                                                    │
│   Step 4: Database Cross-Referencing (UniProt REST API / PDB IDs)      │
│                   │                                                    │
│                   ▼                                                    │
│   Step 5: Publication Visualizations (Exon Map, GC Profile, AA Bar)    │
│                   │                                                    │
│                   ▼                                                    │
│   Step 6: Automated Markdown Report & CSV Export                       │
│                   │                                                    │
│                   ▼                                                    │
│   Step 7: 3D Structure Analysis & Contact Map (Bio.PDB + Distance Mat) │
│                   │                                                    │
│                   ▼                                                    │
│   Step 8: Interactive HTML Gene Card Export                            │
│                   │                                                    │
│                   ▼                                                    │
│   Step 9: Complete OOP Class Assembly (GeneReport Orchestrator)        │
└────────────────────────────────────────────────────────────────────────┘
```


---

## Step 1 · Data Ingestion with API Rate-Limiting & Local Caching

### Specification
Implement `fetch_or_load_data(symbol: str, organism: str, cache_dir: Path, email: str, api_key: Optional[str] = None) -> Bio.SeqRecord.SeqRecord`:
- Check local cache first: `cache_dir / f"{symbol.lower()}_human.gb"`, `cache_dir / f"{symbol}.gb"`, or `DATA_DIR / "genbank" / f"{symbol.lower()}_human.gb"`.
- If missing, query NCBI Entrez with `esearch` + `efetch` respecting rate limits (`time.sleep(0.35)`).
- Cache and return the parsed `SeqRecord`.


In [ ]:
def fetch_or_load_data(
    symbol: str,
    organism: str = "Homo sapiens",
    cache_dir: Path = OUT_DIR / "cache",
    email: str = "student@example.edu",
    api_key: Optional[str] = None
) -> Bio.SeqRecord.SeqRecord:
    # TODO: Fetch or load GenBank record with caching.
    ...


In [ ]:
# ✅ Test Step 1 — Run me to verify
rec_tp53 = fetch_or_load_data("TP53", cache_dir=DATA_DIR / "genbank")
assert rec_tp53.id.startswith("NM_000546")
assert len(rec_tp53.seq) == 2512
print("Step 1 passed ✔")


---

## Step 2 · Genomic Feature Extraction (SeqIO & SeqFeature)

### Specification
Implement `extract_genomic_features(record: Bio.SeqRecord.SeqRecord) -> tuple[pd.DataFrame, Any, str]`:
- Loop over `record.features`. Extract exons (coordinates, length, strand, GC%) and CDS (protein_id, product, translation).
- Return `(features_df, cds_feature, translation_seq)`.


In [ ]:
def extract_genomic_features(
    record: Bio.SeqRecord.SeqRecord
) -> tuple[pd.DataFrame, Any, str]:
    # TODO: Extract exon and CDS annotations into DataFrame.
    ...


In [ ]:
# ✅ Test Step 2 — Run me to verify
df_feat, cds_feat, p_seq = extract_genomic_features(rec_tp53)
assert isinstance(df_feat, pd.DataFrame)
assert len(df_feat[df_feat["type"] == "exon"]) == 11
assert len(p_seq) == 393
print("Step 2 passed ✔")


---

## Step 3 · Biophysical Protein Profiling (ProtParam)

### Specification
Implement `analyze_protein_biophysics(protein_seq: str) -> tuple[ProteinAnalysis, dict[str, Any]]`:
- Clean sequence, initialize `ProteinAnalysis(clean_seq)`.
- Compute: molecular weight (kDa), isoelectric point ($pI$), instability index (<40 stable), aromaticity, GRAVY, secondary structure fractions, amino acid counts, and percentages.


In [ ]:
def analyze_protein_biophysics(protein_seq: str) -> tuple[ProteinAnalysis, dict[str, Any]]:
    # TODO: Compute biophysical metrics using ProteinAnalysis.
    ...


In [ ]:
# ✅ Test Step 3 — Run me to verify
pa_obj, p_metrics = analyze_protein_biophysics(p_seq)
assert 43.0 < p_metrics["molecular_weight_kda"] < 44.0
assert 6.0 < p_metrics["isoelectric_point"] < 6.5
assert p_metrics["is_stable"] is False
assert p_metrics["gravy"] < 0
print("Step 3 passed ✔")


---

## Step 4 · External Knowledge Integration (UniProt REST API)

### Specification
Implement `fetch_uniprot_annotations(symbol: str) -> dict[str, Any]`:
- Query `https://rest.uniprot.org/uniprotkb/search?query=gene_exact:{symbol}+AND+organism_id:9606&format=json`.
- Extract `accession`, `protein_name`, `function`, `subcellular_location`, `pdb_ids`.
- Provide offline fallback for seamless execution without network.


In [ ]:
def fetch_uniprot_annotations(symbol: str) -> dict[str, Any]:
    # TODO: Query UniProt REST API with offline fallback.
    ...


In [ ]:
# ✅ Test Step 4 — Run me to verify
uniprot_info = fetch_uniprot_annotations("TP53")
assert uniprot_info["accession"] == "P04637"
assert "tumor suppressor" in uniprot_info["function"].lower()
assert len(uniprot_info["pdb_ids"]) >= 3
print("Step 4 passed ✔")


---

## Step 5 · Multi-Panel Publication Visualizations

### Specification
Implement `generate_publication_plots(record, features_df, cds_feat, pa_obj, save_dir: Path) -> dict[str, Path]`:
- Generate 3 high-resolution figures:
  1. `exon_structure.png` (coordinate line with exons & CDS)
  2. `gc_profile.png` (sliding window GC% line plot with mean indicator)
  3. `protein_composition.png` (amino acids colored by biochemical class)
- Return dictionary mapping names to saved paths.


In [ ]:
def generate_publication_plots(
    record: Bio.SeqRecord.SeqRecord,
    features_df: pd.DataFrame,
    cds_feat: Any,
    pa_obj: ProteinAnalysis,
    save_dir: Path
) -> dict[str, Path]:
    # TODO: Generate and save exon_structure, gc_profile, and protein_composition plots.
    ...


In [ ]:
# ✅ Test Step 5 — Run me to verify
fig_dict = generate_publication_plots(rec_tp53, df_feat, cds_feat, pa_obj, OUT_DIR / "figures")
assert fig_dict["exon_structure"].exists()
assert fig_dict["gc_profile"].exists()
assert fig_dict["protein_composition"].exists()
print("Step 5 passed ✔")


---

## Step 6 · Automated Markdown Report & Tabular Export

### Specification
Implement `export_markdown_report(symbol, organism, record, features_df, p_metrics, uniprot_info, figures_dict, out_dir: Path) -> Path`:
- Export `features.csv` and `protein_composition.csv`.
- Compile complete Markdown summary report `<GENE>_Gene_Report.md`.


In [ ]:
def export_markdown_report(
    symbol: str,
    organism: str,
    record: Bio.SeqRecord.SeqRecord,
    features_df: pd.DataFrame,
    p_metrics: dict[str, Any],
    uniprot_info: dict[str, Any],
    figures_dict: dict[str, Path],
    out_dir: Path
) -> Path:
    # TODO: Export CSVs and compile full Markdown summary report.
    ...


In [ ]:
# ✅ Test Step 6 — Run me to verify
rep_p = export_markdown_report("TP53", "Homo sapiens", rec_tp53, df_feat, p_metrics, uniprot_info, fig_dict, OUT_DIR)
assert rep_p.exists()
assert (OUT_DIR / "TP53_features.csv").exists()
print("Step 6 passed ✔")


---

## Step 7 · 3D Structural Analysis & Residue Contact Map (Bio.PDB + Distance Matrix)

### Biological Context
A protein's 3D tertiary fold is maintained by non-covalent contacts (hydrophobic packing, salt bridges, hydrogen bonds) between amino acid residues distant in sequence but proximal in 3D space. 

A **Residue Contact Map** is an $(N \times N)$ symmetric binary matrix representing whether residue $i$ and residue $j$ have alpha-carbon ($C_\alpha$) distance $d_{ij} < 8.0\text{ Å}$.
Secondary structures produce distinct contact patterns:
- $\alpha$-helices form contacts along the immediate diagonal ($i, i+4$).
- Antiparallel $\beta$-sheets form perpendicular contact diagonals.

### Specification
Implement `analyze_structure_contact_map(pdb_filepath: Path, chain_id: str = "A", contact_cutoff: float = 8.0, save_path: Path = None) -> tuple[np.ndarray, dict[str, Any]]`:
1. Parse `pdb_filepath` using Biopython's `Bio.PDB.PDBParser(QUIET=True)`.
2. Extract all alpha carbon ($C_\alpha$) atoms for the specified `chain_id`.
3. Extract 3D coordinates into a $(N \times 3)$ NumPy array.
4. Compute the $(N \times N)$ Euclidean distance matrix using NumPy array broadcasting:
   `dist_mat = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)`
5. Define contacts: `contacts = (dist_mat < contact_cutoff) & (dist_mat > 0.0)`.
6. If `save_path` is given, plot and save the contact map.
7. Return `(contacts_matrix, structural_metrics_dict)`.


In [ ]:
def analyze_structure_contact_map(
    pdb_filepath: Path,
    chain_id: str = "A",
    contact_cutoff: float = 8.0,
    save_path: Path = None
) -> tuple[np.ndarray, dict[str, Any]]:
    # TODO: Parse PDB, compute vectorized pairwise distance matrix, generate contact map.
    ...


In [ ]:
# ✅ Test Step 7 — Run me to verify
pdb_file = DATA_DIR / "structures" / "1tup.pdb"
cmap_png = OUT_DIR / "figures" / "contact_map.png"
contacts_m, struct_m = analyze_structure_contact_map(pdb_file, chain_id="A", save_path=cmap_png)

print("Structural Metrics (1TUP Chain A):", struct_m)
assert contacts_m.shape == (196, 196)
assert struct_m["num_residues"] == 196
assert struct_m["total_contacts"] > 500
assert cmap_png.exists()
print("Step 7 passed ✔")


---

## Step 8 · Interactive HTML Gene Card Export

### Biological Context
In clinical genomics and translational research, reports are often distributed as self-contained **HTML Gene Cards** that researchers can open in any browser without external dependencies. 

### Specification
Implement `export_html_gene_card(symbol, organism, record, p_metrics, uniprot_info, figures_dict, struct_metrics, out_dir: Path) -> Path`:
- Construct a single standalone HTML document `<GENE>_gene_card.html` with:
  - Clean embedded CSS (card design, badges, responsive tables).
  - Key parameter summary badges.
  - Base64 or relative image embeddings of all 4 generated figures.
  - Direct links to NCBI, UniProt, and RCSB PDB databases.
- Save to `out_dir / f"{symbol}_gene_card.html"` and return the `Path`.


In [ ]:
def export_html_gene_card(
    symbol: str,
    organism: str,
    record: Bio.SeqRecord.SeqRecord,
    p_metrics: dict[str, Any],
    uniprot_info: dict[str, Any],
    figures_dict: dict[str, Path],
    struct_metrics: dict[str, Any],
    out_dir: Path
) -> Path:
    # TODO: Build self-contained HTML Gene Card and export to disk.
    ...


In [ ]:
# ✅ Test Step 8 — Run me to verify
card_p = export_html_gene_card(
    "TP53", "Homo sapiens", rec_tp53, p_metrics,
    uniprot_info, fig_dict, struct_m, OUT_DIR
)
assert card_p.exists()
assert "Executive Biomolecular Summary" in card_p.read_text()
print(f"Gene card exported to: {card_p.name}")
print("Step 8 passed ✔")


---

## Step 9 · Complete OOP Encapsulation: The `GeneReport` Class

### Specification
Assemble all modular pipeline steps into a unified `GeneReport` class:
- Attributes: protected `_symbol`, `_record`, `_features_df`, `_protein_seq`, `_p_metrics`, `_uniprot_data`, `_struct_metrics`, `_figures`.
- Properties: `.symbol`, `.accession`, `.sequence_length`, `.gc_content`, `.num_exons`, `.protein_id`, `.molecular_weight_kda`, `.is_stable`.
- Method `run()`: Executes Steps 1 through 8 end-to-end and returns a comprehensive execution manifest.


In [ ]:
class GeneReport:
    """
    Comprehensive automated pipeline class for gene reporting.
    """
    def __init__(
        self,
        symbol: str = "TP53",
        organism: str = "Homo sapiens",
        cache_dir: Optional[Path] = None,
        out_dir: Optional[Path] = None
    ):
        # TODO: Initialize attributes
        ...
        
    # TODO: Implement properties and pipeline methods


In [ ]:
# ✅ Final Capstone Test — Run Pipeline End-to-End
pipeline = GeneReport(symbol="TP53", organism="Homo sapiens")
manifest = pipeline.run()

print("\n--- Pipeline Result Manifest ---")
for k, v in manifest.items():
    if k != "figures":
        print(f"  {k:20s}: {v}")

assert pipeline.symbol == "TP53"
assert pipeline.accession == "NM_000546.6"
assert pipeline.sequence_length == 2512
assert pipeline.num_exons == 11
assert manifest["markdown_report"].exists()
assert manifest["html_card"].exists()
assert "contact_map" in pipeline._figures
print("\nCapstone Project Pipeline verification PASSED successfully! 🎓🎉")


---

## 8. Summary & Capstone Architecture Review

| Component | Modules Applied | Method / Task |
|---|---|---|
| **Data Ingestion** | M02, M05A, M05C | `fetch_or_load_data()` with offline caching & Entrez |
| **Feature Extraction** | M01, M02C | `extract_genomic_features()` for Exons & CDS |
| **Protein Profiling** | M01, M02C | `analyze_protein_biophysics()` for MW, pI, Instability, GRAVY |
| **Database Cross-Ref** | M05B | `fetch_uniprot_annotations()` via UniProt REST API |
| **Visualizations** | M04B, M04C, M06 | `generate_publication_plots()` (Exon map, GC profile, AA bar) |
| **3D Structural Analysis**| M02C, M06 | `analyze_structure_contact_map()` via `Bio.PDB` distance matrix |
| **Automated Reports** | M02A, M06 | `export_markdown_report()` & `export_html_gene_card()` |
| **Pipeline OOP Class** | M03A, M03B, M03C | `GeneReport` class orchestrator |
